In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
path = "IndianOcean.nc"
ds = xr.open_dataset(path)

In [ ]:
thetao_glor_mean = ds['thetao_glor'].mean(dim=['time', 'latitude', 'longitude'], skipna=True)
thetao_glor_mean

In [ ]:
import xarray as xr
import numpy as np
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt

# 提取变量
thetao_glor = ds['thetao_glor']  # 温度变量
depths = ds['depth'].values

# 计算经纬度和时间的平均值
temperature_mean = thetao_glor.mean(dim=["latitude", "longitude", "time"]).values

# 标记 NaN 层
is_nan = np.isnan(temperature_mean)

# 分离有效层和 NaN 层
valid_indices = ~is_nan
valid_temperature = temperature_mean[valid_indices]
valid_depths = depths[valid_indices]

# 如果有效数据不足，直接退出
if len(valid_temperature) < 2:
    raise ValueError("有效数据不足，无法进行聚类。")

# 计算有效层的成对距离
distance_matrix = pdist(valid_temperature[:, np.newaxis], metric='euclidean')

# 检查距离矩阵是否有限
if np.any(~np.isfinite(distance_matrix)):
    raise ValueError("Non-finite values in distance matrix.")

# 进行层次聚类
linkage_matrix = sch.linkage(distance_matrix, method='ward')

# 获取聚类标签
num_clusters = 5  # 设置聚类数
valid_cluster_labels = sch.fcluster(linkage_matrix, num_clusters, criterion='maxclust')

# 初始化完整的标签数组
full_labels = np.zeros_like(depths, dtype=int)

# 为有效层分配聚类标签
full_labels[valid_indices] = valid_cluster_labels

# 为 NaN 层分配最近的类别
for i, is_nan_layer in enumerate(is_nan):
    if is_nan_layer:
        # 获取最近的邻近层索引
        previous_label = full_labels[i - 1] if i > 0 else None
        next_label = full_labels[i + 1] if i < len(full_labels) - 1 else None
        
        # 分配类别
        if previous_label and next_label:
            # 与最近的有效层同类
            full_labels[i] = previous_label if previous_label == next_label else next_label
        elif previous_label:
            full_labels[i] = previous_label
        elif next_label:
            full_labels[i] = next_label
        else:
            full_labels[i] = num_clusters + 1  # 单独分为新类（应避免）


In [ ]:
# 获取深度对应的聚类标签
depth_labels = full_labels

# 找出聚类标签和对应的最浅深度
unique_clusters = np.unique(depth_labels)
cluster_depth_mapping = {
    cluster: np.min(np.where(depth_labels == cluster)[0]) for cluster in unique_clusters
}

# 按深度从浅到深对聚类编号排序
sorted_clusters = sorted(cluster_depth_mapping.items(), key=lambda x: x[1])
new_cluster_mapping = {cluster: idx + 1 for idx, (cluster, _) in enumerate(sorted_clusters)}

# 应用新的类别编号
renamed_clusters = np.array([new_cluster_mapping[label] for label in depth_labels])

# 打印聚类结果
for depth, original_label, renamed_label in zip(depths, full_labels, renamed_clusters):
    print(f"Depth: {depth}, Original Cluster: {original_label}, Renamed Cluster: {renamed_label}")

# 可视化树状图
plt.figure(figsize=(10, 6))
sch.dendrogram(linkage_matrix, labels=valid_depths)
plt.title("Hierarchical Clustering Dendrogram (Depth)")
plt.xlabel("Depth")
plt.ylabel("Cluster Distance")
plt.show()

In [ ]:
if len(renamed_clusters.shape) == 1:
    # 将 1D 的 renamed_clusters 广播到 3D
    renamed_clusters = np.tile(renamed_clusters[:, np.newaxis, np.newaxis], 
                                (1, len(ds['latitude']), len(ds['longitude'])))

# 创建一个 DataArray 来保存新聚类标签
renamed_cluster_dataarray = xr.DataArray(
    renamed_clusters,
    dims=['depth', 'latitude', 'longitude'],
    coords={'depth': ds['depth'], 'latitude': ds['latitude'], 'longitude': ds['longitude']},
    name='cluster_labels'
)

# 将新变量添加到原始数据集中
ds['cluster_labels'] = renamed_cluster_dataarray